# Module 02 Homework - Pandas
Using a dataset from _Wine Spectator_, a wine magazine, we will practice the material that we covered in notebook M02_Colab04 to M02_Colab06. This notebook will cover data transformation, grouping, and sorting using pandas.

Created by Angie Avalos Joel (015752230)  
Last updated: 02/22/2026

In [1]:
csvurl="https://gist.githubusercontent.com/clairehq/" + \
        "79acab35be50eaf1c383948ed3fd1129/raw/407a02139ae1e134992b90b4b2b8c329b3d73a6a/winemag-data-130k-v2.csv"
import pandas as pd
wine = pd.read_csv(csvurl)

**Data cleaning**  
Notice that the first column is redundant. Part of data analysis is cleaning and removing redundancy. How would you drop the redundant column inplace, that is overwrite the dataframe.

In [5]:
wine.head(1)

,Unnamed: 0,country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery
0,0,Italy,"Aromas include tropical fruit, broom, brimston...",Vulkà Bianco,87,NaN,Sicily & Sardinia,Etna,NaN,Kerin O’Keefe,@kerinokeefe,Nicosia 2013 Vulkà Bianco (Etna),White Blend,Nicosia


In [25]:
wine.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65499 entries, 0 to 65498
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Unnamed: 0             65499 non-null  int64  
 1   country                65467 non-null  object 
 2   description            65499 non-null  object 
 3   designation            46588 non-null  object 
 4   points                 65499 non-null  int64  
 5   price                  60829 non-null  float64
 6   province               65467 non-null  object 
 7   region_1               54744 non-null  object 
 8   region_2               25170 non-null  object 
 9   taster_name            51856 non-null  object 
 10  taster_twitter_handle  49467 non-null  object 
 11  title                  65499 non-null  object 
 12  variety                65499 non-null  object 
 13  winery                 65499 non-null  object 
dtypes: float64(1), int64(2), object(11)
memory usage: 7.0+

In [4]:
wine_dropped = wine.drop(['Unnamed: 0'], axis=1)
wine_dropped.head()

,country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery
0,Italy,"Aromas include tropical fruit, broom, brimston...",Vulkà Bianco,87,NaN,Sicily & Sardinia,Etna,NaN,Kerin O’Keefe,@kerinokeefe,Nicosia 2013 Vulkà Bianco (Etna),White Blend,Nicosia
1,Portugal,"This is ripe and fruity, a wine that is smooth...",Avidagos,87,15.0,Douro,NaN,NaN,Roger Voss,@vossroger,Quinta dos Avidagos 2011 Avidagos Red (Douro),Portuguese Red,Quinta dos Avidagos
2,US,"Tart and snappy, the flavors of lime flesh and...",NaN,87,14.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Rainstorm 2013 Pinot Gris (Willamette Valley),Pinot Gris,Rainstorm
3,US,"Pineapple rind, lemon pith and orange blossom ...",Reserve Late Harvest,87,13.0,Michigan,Lake Michigan Shore,NaN,Alexander Peartree,NaN,St. Julian 2013 Reserve Late Harvest Riesling ...,Riesling,St. Julian
4,US,"Much like the regular bottling from 2012, this...",Vintner's Reserve Wild Child Block,87,65.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Sweet Cheeks 2012 Vintner's Reserve Wild Child...,Pinot Noir,Sweet Cheeks


#### Question 1: ####  
What is the mean of the points column?

In [9]:
wine['points'].mean()

np.float64(88.43403716087269)

#### Question 2: ####  
How many countries are present in this dataset? (Only count each country once)

In [15]:
wine['country'].nunique()

41

#### Question 3: ####
How many times does each country appeared in this dataset? Show each country and the corresponding count (show counts in ascending order)

In [17]:
wine['country'].value_counts().sort_values(ascending=True)

,count
country,
Bosnia and Herzegovina,1
Slovakia,1
Armenia,1
Luxembourg,4
Switzerland,4
India,4
Ukraine,5
Macedonia,6
Czech Republic,6


#### Question 4: ####
Create a variable `adjusted_price` containing the adjusted price which is the price subtracted by the average price. *This is called **"centering" transformation** - a method commonly used in the preprocessing step before applying various machine learning algorithms.*

In [20]:
adjusted_price = wine['price'] - (wine['price'].mean())
adjusted_price

,price
0,NaN
1,-20.232932
2,-21.232932
3,-22.232932
4,29.767068
...,...
65494,9.767068
65495,-13.232932
65496,-15.232932
65497,-4.232932


#### Question 5: ####
What is the title of the wine that has the highest points-to-price ratio in the dataset?

In [24]:
wine.loc[(wine['points']/wine['price']).idxmax(), 'title']

'Bandit NV Merlot (California)'

#### Question 6: ####
Create a series `flavor_counts` that contains two values: the number of wines that has the word "tart" in the `description` column and the number of wines that has the word "berries" in the `description` column. The index of the Series should be "Tart" and "Berries" for the corresponding values.

In [27]:
# Look for the word "tart", not case sensitive, not a substring of another word.
# Same applies to the word "berries"
flavor_counts = pd.Series(
    [
    wine['description'].map(lambda x: 'tart' in x.lower()).sum(),
    wine['description'].map(lambda x: 'berries' in x.lower()).sum()
    ],
    index = ['Tart', 'Berries']
)

flavor_counts

,0
Tart,4424
Berries,3512


#### Question 7: ####
Let's convert the points into simple star ratings. A score of 90 or higher counts as 3 stars, a score of at least 80 but less than 90 is 2 stars. Any other score is 1 star.

Also, any wines from France should automatically get 3 stars, regardless of points.

Add this new column `star_ratings` to the dataframe with the number of stars for each wine in the dataset.

In [33]:
wine['star_ratings']= wine['points'].apply(lambda x: 3 if x >= 90 else (2 if x >= 80 else 1))
wine.loc[wine['country'] == 'France', 'star_ratings'] = 3
wine.tail()

,Unnamed: 0,country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery,star_ratings
65494,65494,France,Made from young vines from the Vaulorent porti...,Fourchaume Premier Cru,90,45.0,Burgundy,Chablis,NaN,Roger Voss,@vossroger,William Fèvre 2005 Fourchaume Premier Cru (Ch...,Chardonnay,William Fèvre,3
65495,65495,Australia,"This is a big, fat, almost sweet-tasting Caber...",NaN,90,22.0,South Australia,McLaren Vale,NaN,Joe Czerwinski,@JoeCz,Tapestry 2005 Cabernet Sauvignon (McLaren Vale),Cabernet Sauvignon,Tapestry,3
65496,65496,US,"Much improved over the unripe 2005, Fritz's 20...",Estate,90,20.0,California,Dry Creek Valley,Sonoma,NaN,NaN,Fritz 2006 Estate Sauvignon Blanc (Dry Creek V...,Sauvignon Blanc,Fritz,3
65497,65497,US,This wine wears its 15.8% alcohol better than ...,Block 24,90,31.0,California,Napa Valley,Napa,NaN,NaN,Hendry 2004 Block 24 Primitivo (Napa Valley),Primitivo,Hendry,3
65498,65498,Spain,"A unique take on Manzanilla Sherry, which is o...",Manzanilla,90,10.0,Andalucia,Jerez,NaN,Michael Schachner,@wineschach,Bodegas Dios Baco S.L. NV Manzanilla Sherry (J...,Sherry,Bodegas Dios Baco S.L.,3


#### Question 8: ####
Who are the most common wine reviewers in the dataset? Create a Series whose index is the taster_twitter_handle category from the dataset, and whose values count how many reviews each person wrote.

In [35]:
wine['taster_twitter_handle'].value_counts()


,count
taster_twitter_handle,
@vossroger,13045
@wineschach,7752
@kerinokeefe,5313
@paulgwine,4851
@vboone,4696
@mattkettmann,3035
@JoeCz,2605
@wawinereport,2358
@gordone_cellars,2032


#### Question 9: ####
What combination of countries and varieties are most common? Create a Series whose index is a MultiIndexof {country, variety} pairs. For example, a pinot noir produced in the US should map to {"US", "Pinot Noir"}. Sort the values in the Series in descending order based on wine count.

In [42]:
wine.groupby(['country', 'variety']).size().sort_values(ascending=False)

country    variety                 
US         Pinot Noir                  4918
           Cabernet Sauvignon          3649
           Chardonnay                  3412
France     Bordeaux-style Red Blend    2380
Italy      Red Blend                   1870
                                       ... 
Romania    Rosé                           1
US         Ugni Blanc                     1
           Touriga                        1
           Torrontés                      1
Argentina  Merlot-Cabernet Franc          1
Length: 1304, dtype: int64

#### Question 10 #####
Create a Series whose index is reviewers and whose values is the average score given out by that reviewer.  
*Hint:* You will need the `taster_name` and `points` columns.

In [43]:
wine.groupby('taster_name')['points'].mean()

,points
taster_name,
Alexander Peartree,86.014286
Anna Lee C. Iijima,88.380506
Anne Krebiehl MW,90.587903
Carrie Dykes,86.644444
Christina Pickard,89.500000
Fiona Adams,87.090909
Jeff Jenssen,88.273504
Jim Gordon,88.604331
Joe Czerwinski,88.519770
